**r = 16**
원본 모델의 행렬 가중치를, 작은 행렬로 쪼개어 연산

**q_proj(query)** 현재 읽고 있는 단어의 정보를 담는 주체

**k-proj(key)** 문장 내의 단어들이 query와 어떤 연관을 가지는지 비교하는 대상

**v_proj(value)** 단어들의 실제 의미 값이 담긴 행렬

**o_proj(output)** 어텐션 연산 결과를 다음 신경망 층으로 전달하기 위해 정돈하는 행렬

In [9]:
# Unsloth 및 필수 라이브러리 설치
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes

from unsloth import FastLanguageModel
import torch

# Llama 3.2 3B 모델 및 Tokenizer 로드
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct", # Meta 공식 3B Instruct 모델
    max_seq_length = 2048, # 한 번에 처리할 수 있는 토큰 수
    load_in_4bit = True, # 4bit 수준으로 압축하여 load
)


# LoRA(양자화 학습) 설정 적용
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # rank, 차원
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"], # query, key, value, output 모두 타겟팅
    lora_alpha = 16, # 학습 가중치
    lora_dropout = 0, # overfitting 방지 비율
    bias = "none", # bias 학습 여부
)

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [10]:
import json
import random
from datasets import Dataset

# 티켓 이미지 기반 OCR 데이터와 정답 데이터 매핑
base_data = [
    {
        "raw_text": "티켓링그 | Someday Festival 2024 | Someday Festival 2024 | 일시\n: 2024년 09월 07일(토) 12:00 | 704716941 | 장소 : 난지한강공원 | 장소 : 난지한강공원\n이민준 | 010-****-4556 | 입장권 | 일일권(토) | 119.000원 |\n* 티켓 분실 시 어떠한 사유에도 재발행이 불가하오니. 보관에 각별히 주의하시기 바람니다.\n▣ 주최 : ㈜티브이조선미디어렙 | ▣ 문의 : 1544-1813 | 2024/O7/11 |\n20240820-NTL0-00-744 (1/1) | 온라인 파트너 | NHN LINK | 네이벌예약 포인트",
        "json_output": {"title": "Someday Festival 2024", "date": "2024-09-07", "time": "12:00", "location": "난지한강공원", "seat": "일일권(토)", "platform": ["티켓링크"], "price": 119000}
    },
    {
        "raw_text": "Elegy | 2024 HA HYUN SANG CONCERT | 일 시 : 2024년 12월 01일(일) 오후 5:OO |\n장 소 : 올림픽공원 올림픽핸드볼경기장 | A석 | 2층 41구역 54번 |\n예약번화: T2570988160 (1/1) | 예 매 자 : **0320 [이민준 님] |\n전화번호: 010-****-4556 | 일반 | 금액: 132.000원 | 판매일자: 2024-10-02 |\n*본 티겟 분실, 도난, 미소지시 재발행되지 안으며 공연관람이 불가함니다 보관에 유의 바람\n*주최/주관: WAKE ONE ㈜CJ ENM | HA HYUN SANG | 콘서트",
        "json_output": {"title": "2024 HA HYUN SANG CONCERT Elegy", "date": "2024-12-01", "time": "17:00", "location": "올림픽공원 올림픽핸드볼경기장", "seat": "A석 2층 41구역 54번", "platform": [], "price": 132000}
    },
    {
        "raw_text": "티켓링쿠 | 김두루미 vol.04 (kimdurumi vol.04) | 일시 : 2024년 11월 09일(토) 14:00 |\n1473643535 | 장소 : 서울대학교 제1파워플랜트 | 이민준 | 010-****-4556 |\n비지정석 | 일반권 | 88,000원 | ▣ 주최 : 예헤헤 (EHEHE), 백승렬 |\n▣ 주관 : 예헤헤 (EHEHE) | ▣ 문의 : 예헤헤 (EHEHE) | 2024/10/04 |\n20241108-A24901-98-274 (1/1) | 티켓링쿠앱 | 2024/11/08 17:05:55 | PAYCO 포인트 | NHN L1NK",
        "json_output": {"title": "김두루미 vol.04 (kimdurumi vol.04)", "date": "2024-11-09", "time": "14:00", "location": "서울대학교 제1파워플랜트", "seat": "비지정석 일반권", "platform": ["티켓링크"], "price": 88000}
    },
    {
        "raw_text": "interpark티켓 | Lacuna (라쿠나) 단독 콘서트 ' dream:parallel world ' |\n일 시 : 2024년 10월 03일(목) 오후 5:00 | 장 소 : YES24 L1VE HALL |\n지정석 | 2층 X구역 3열 13번 | 예약번호: T2552107370 (1/1) |\n예 매 자 : **0320 [이민준 님] | 전화번호: 010-****-4556(59) | 금액: 99.000 |\n판매일자: 2024-09-12 | 주최: 엔피엠지 뮤직 | 주관: 주식회사 엔피엠지 | 후원: CJ문화재단",
        "json_output": {"title": "Lacuna (라쿠나) 단독 콘서트 ' dream:parallel world '", "date": "2024-10-03", "time": "17:00", "location": "YES24 LIVE HALL", "seat": "지정석 2층 X구역 3열 13번", "platform": ["인터파크 티켓"], "price": 99000}
    },
    {
        "raw_text": "NTEERS ASIA TOUR 2024 JANGCHUNG ARENA |\n일 시 : 2024년 08월 30일(금) 오후 8:00 | 장 소 : 장충체육관 | 스텐딩석 |\n1층 스텐딩A 입장번호 297번 | 예약번호: T2522946930 (1/1) |\n예 매 자 : **0320 [이민준 님] | 전화번호: 010-****-4556 | 금액: 132,000 |\n주최/주관 : 파이아일랜드레코즈, 주식회사 인터파크트리플 | 문의 : 1544-1555",
        "json_output": {"title": "THE VOLUNTEERS ASIA TOUR 2024", "date": "2024-08-30", "time": "20:00", "location": "장충체육관", "seat": "스탠딩석 1층 스탠딩A 입장번호 297번", "platform": [], "price": 132000}
    },
    {
        "raw_text": "빨래는 오늘을 살아가는 우리들의 이야기다 | 2024-O7-17(수) 오후 3시00분 |\n인터파크 유니플렉스 2관 | 2층 2열 8번 | T2504176540 (1/1) |\n학생 할인(중,고,대학생/본인만)25% | 기획·제작: ㈜씨에이치수박 | 문의: 02-928-3362 |\nR석 | 57.750원 | 이민준(4556) | MUSICAL 빨래 | 빨래",
        "json_output": {"title": "뮤지컬 빨래", "date": "2024-07-17", "time": "15:00", "location": "인터파크 유니플렉스 2관", "seat": "R석 2층 2열 8번", "platform": [], "price": 57750}
    },
    {
        "raw_text": "interpark 티켓 | SURL concert ' ? YRU ? ' |\n일 시 : 2024년 04월 28일 일요일 오후 6시 00분 | 장 소 : 무신사 개러지 |\n전석 77,000 | 입장번호 38 | 주최: 엔피엠지 뮤직 | 주관: 주식회사 엔피엠지 | SURL",
        "json_output": {"title": "SURL concert ' ? YRU ? '", "date": "2024-04-28", "time": "18:00", "location": "무신사 개러지", "seat": "입장번호 38", "platform": ["인터파크 티켓"], "price": 77000}
    },
    {
        "raw_text": "Christmas in NELL'S ROOM 2025 | 일 시 : 2025년 12월 25일(목) 오후 7:OO |\n장 소 : 잠실 학생체육관 | R석 | FLOOR 나구역 13열 19번 |\n예약번호: T2847532270 (1/1) | 예 매 자 : **0320 [이민준 님] |\n전화번호: 010-****-4556 | 일반 | 금액: 165.000 | 판매일자: 2O25-11-O4 |\n주최: ㈜스페이스보헤미안 | SPACE BOHEMIAN | 165,000원",
        "json_output": {"title": "Christmas in NELL'S ROOM 2025", "date": "2025-12-25", "time": "19:00", "location": "잠실 학생체육관", "seat": "R석 FLOOR 나구역 13열 19번", "platform": ["NOL 티켓"], "price": 165000}
    },
    {
        "raw_text": "AoB Breakout 2025: The Final | 일시 : 2025-12-11 19:OO |\n장소 : 무신사 개러지 (Musinsa Garage) | [스텐딩석] 160번 |\n예약번호 : T25120423593134-33351 | 예매자 : 이*준 |\n금액 : 1.000 | 결제수단 : 간편결제 | 예매일자 : 2025-12-04 23:59 |\n주최/기획 AoB, 호라이즌 (HORIZON) | 29CM | 1000원",
        "json_output": {"title": "AoB Breakout 2025: The Final", "date": "2025-12-11", "time": "19:00", "location": "무신사 개러지 (Musinsa Garage)", "seat": "[스탠딩석] 160번", "platform": ["29CM"], "price": 1000}
    },
    {
        "raw_text": "백예린 2025 Live <wanna see you dance again> |\n일 시: 2025년 11월 20일(목) 오후 8:00 | 장 소: 에스팩토리 |\n예약번호: T2849954420 (1/1) | 예 매 자 : **O320 [이민준 님] |\n전화번호: 010-****-4556 | 스텐딩 | A구역 입장번호 493번 | 일반 |\n금액: 110,000원 | 판매일자: 2O25-11-O7 | 주최/주관: 피플라이크피플 | NOL ticket",
        "json_output": {"title": "백예린 2025 Live 〈wanna see you dance again〉", "date": "2025-11-20", "time": "20:00", "location": "에스팩토리", "seat": "스탠딩 A구역 입장번호 493번", "platform": ["NOL 티켓"], "price": 110000}
    },
    {
        "raw_text": "2025 렛츠락 페스티벌 | Lets Rock Festival |\n일 시: 2025년 09월 06일(토) 오후 12:00 | 장 소: 난지 한강공원 일대 |\n1일권 비지정석 | 예약번호: T2786724091 (2/2) | 예 매 자 : **0320 |\n금액: 110.000원 (일반) | 판매일자: 2025-08-O7 | NOL ticket | NOL ticket",
        "json_output": {"title": "2025 렛츠락 페스티벌", "date": "2025-09-06", "time": "12:00", "location": "난지 한강공원 일대", "seat": "1일권 비지정석", "platform": ["NOL 티켓"], "price": 110000}
    },
    {
        "raw_text": "입장번호 140 번 | RAP HOUSE VOL.35 | 랩하우스 | RAP HOUSE VOL.35 | interpark티켓 | 2024년 11월 15일(금) 오후 8:30 | 플렉스 라운지 | 일 시: 2024년 11월 15일(금) 오후 8:3O | 장 소: 플렉스 라운지 | 전석 | 입장번호 140 번 | 일반 | 33,000원 | 카카오페이 |\n예 매 자 : **0715 [이현빈 님] | 전화번호: O1u-****-7559 | 금액: 33.000원 | 판매일자: 2024-11-12 | 예매처 : 인터피크 모바일 | 주최: RAP HOUSE",
        "json_output": {"title": "랩하우스 RAP HOUSE VOL.35", "date": "2024-11-15", "time": "20:30", "location": "플렉스 라운지", "seat": "입장번호 140번", "platform": ["인터파크 티켓"], "price": 33000}
    }
]

# OCR 임의 노이즈 추가 주입 함수
def inject_random_noise(text):
    decorations = ["|", "[", "]", "▣", "*", "-", "/", "  "]
    replacements = {"0": "O", "1": "L", "5": "S", "8": "B", "합니다": "함니다"}

    words = text.split(" ")
    noised_words = []
    for word in words:
        for src, tgt in replacements.items():
            if src in word and random.random() < 0.4:
                word = word.replace(src, tgt)
        if random.random() < 0.1:
            word = f"\n{word}"
        if random.random() < 0.1:
            word = f"{word} {random.choice(decorations)}"
        noised_words.append(word)
    return " ".join(noised_words)

# 데이터 증강 (12개 조합 * 5개씩 무작위 생성 = 총 60개 데이터셋)
augmented_inputs = []
augmented_outputs = []

for item in base_data:
    for _ in range(5):
        noisy_text = inject_random_noise(item["raw_text"])
        clean_json_str = json.dumps(item["json_output"], ensure_ascii=False)
        augmented_inputs.append(noisy_text)
        augmented_outputs.append(clean_json_str)

# Unsloth 전용 프롬프트 포맷팅
ticket_prompt = """너는 공연 티켓 분석 전문가야. 다음 지저분한 OCR 텍스트에서 정보를 추출해 JSON으로만 답해줘.

### 지시사항:
- 추출 항목: 공연명(title), 날짜(date), 시간(time), 장소(location), 좌석 정보(seat), 예매처(platform), 가격(price)
- 날짜는 YYYY-MM-DD 형식을 엄격히 지키고, 가격은 숫자(Integer) 형태로만 저장해.

### OCR 텍스트:
{}

### 결과 JSON:
{}"""

EOS_TOKEN = tokenizer.eos_token
formatted_texts = []

for inp, out in zip(augmented_inputs, augmented_outputs):
    text = ticket_prompt.format(inp, out) + EOS_TOKEN
    formatted_texts.append(text)

# Hugging Face Dataset 객체 생성 (SFTTrainer 입력 규격)
dataset = Dataset.from_dict({"text": formatted_texts})

print("--- 데이터셋 생성 완료! 총 샘플 수:", len(dataset), "---")
print(dataset[0]["text"])

--- 데이터셋 생성 완료! 총 샘플 수: 60 ---
너는 공연 티켓 분석 전문가야. 다음 지저분한 OCR 텍스트에서 정보를 추출해 JSON으로만 답해줘.

### 지시사항:
- 추출 항목: 공연명(title), 날짜(date), 시간(time), 장소(location), 좌석 정보(seat), 예매처(platform), 가격(price)
- 날짜는 YYYY-MM-DD 형식을 엄격히 지키고, 가격은 숫자(Integer) 형태로만 저장해.

### OCR 텍스트:

티켓링그 | Someday Festival 2024 | Someday Festival 2O24 | ] 일시
: 2024년 09월 07일(토)    
12:OO | 7047L694L ] | 장소 [ : 
난지한강공원 ] | 장소 : [ 난지한강공원
이민준 | / 0L0-****-4SS6 | 입장권 ] | 일일권(토) | 119.000원 |
* 티켓 분실 시 어떠한 사유에도 재발행이 불가하오니. 보관에 각별히 주의하시기 바람니다.
▣ * 주최 : ㈜티브이조선미디어렙 | ▣ 문의 : 1S44-1813 | 2024/O7/11 |
20240820-NTL0-00-744 (L/L) * | 온라인 파트너 | NHN LINK | 
네이벌예약 포인트

### 결과 JSON:
{"title": "Someday Festival 2024", "date": "2024-09-07", "time": "12:00", "location": "난지한강공원", "seat": "일일권(토)", "platform": ["티켓링크"], "price": 119000}<|eot_id|>


In [11]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

# SFTTrainer 주입 및 설정
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,       # Cell 2에서 만든 60개의 데이터셋 변수명과 일치해야 합니다.
    dataset_text_field = "text",   # Cell 2에서 지정한 딕셔너리 Key인 "text"와 일치해야 합니다.
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,     # 한 번에 2개씩 처리 (T4 GPU 메모리 최적화)
        gradient_accumulation_steps = 4,     # 4번 모아서 가중치 업데이트 (실제 배치 사이즈 = 2 * 4 = 8)
        warmup_steps = 5,
        max_steps = 60,                      # 총 60번의 학습 스텝 진행 (60개 데이터 기준 약 8에포크 학습)
        learning_rate = 2e-4,                # LoRA 파인튜닝 권장 학습률
        fp16 = not torch.cuda.is_bf16_supported(), # GPU 가속 활성화
        logging_steps = 1,                   # 매 스텝마다 학습 로그(Loss) 출력
        output_dir = "outputs",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/60 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 60 | Num Epochs = 8 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 9,175,040 of 3,221,924,864 (0.28% trained)


Step,Training Loss
1,3.329666
2,3.391959
3,3.307911
4,3.253491
5,3.243197
6,3.229880
7,3.098321
8,3.043421
9,2.842938
10,2.782464


TrainOutput(global_step=60, training_loss=1.7743560254573822, metrics={'train_runtime': 186.3154, 'train_samples_per_second': 2.576, 'train_steps_per_second': 0.322, 'total_flos': 2919162521346048.0, 'train_loss': 1.7743560254573822, 'epoch': 7.533333333333333})

In [14]:
# TEST
from unsloth import FastLanguageModel

# 모델을 Test 모드로 전환
FastLanguageModel.for_inference(model)

# Prompt
ticket_prompt = """너는 공연 티켓 분석 전문가야. 다음 지저분한 OCR 텍스트에서 정보를 추출해 JSON으로만 답해줘.

### 지시사항:
- 추출 항목: 공연명(title), 날짜(date), 시간(time), 장소(location), 좌석 정보(seat), 예매처(platform), 가격(price)
- 날짜는 YYYY-MM-DD 형식을 엄격히 지키고, 가격은 숫자(Integer) 형태로만 저장해.

### OCR 텍스트:
{}

### 결과 JSON:
{}"""

# test data
heavy_noised_text = """
B구역 입장번호 4O3번 | INTERPARK
2O14 서태지 9집 컴 | 2O14 서태지 9집 컴백 콘서트
백 콘서트 | “크리스마로원”
2O14년 1O월 18일(토) 6:OO p.m | 일시 : 2O14년 1O월 18일(토) 6:OO p.m
잠실 종합운동장 주경기장 | 장소 : 잠실 종합운동장 주경기장
스탠딩 R석 | 스탠딩 R석
정가(143,OOO원) | B구역 입장번호 4O3번
무통장 입금
예약번호: ************* (Web-유소민 님)
예매자 : **1116(Web-유소민 님) | 금액: 정가(143,OOO원)
결제수단: 무통장 입금
전화번호: O1O-****-**** | 판매일자: 2O14/O9/17
예약번호: 1***********
판매처: Web | 주최,주관: 웰메이드쇼21 / 제작,연출: (주)서태지컴퍼니
[유소민 님] | 후원: MBC / 투자: (주)밸류인베스트코리아, CJ E&M, 인터파크INT
문의: 1588-14O7
"""

# 프롬프트 포맷팅
inputs = tokenizer(
    [
        ticket_prompt.format(heavy_noised_text, "")
    ],
    return_tensors = "pt"
).to("cuda")

# 모델 실행
outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)

print("TEST 결과")
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(decoded_output.split("### 결과 JSON:\n")[-1])

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


TEST 결과
{"title": "2024 서태지 9집 백 콘서트 '크리스마로원'", "date": "2024-10-18", "time": "18:00", "location": "잠실 종합운동장 주경기장", "seat": "스탠딩 R석", "platform": ["예매번호: ************* (Web-유소민 님)"], "price": 143000}


In [13]:
# 학습된 LoRA 가중치 저장 (백업용)
model.save_pretrained("ticket_model_lora")
tokenizer.save_pretrained("ticket_model_lora")

# 원본 모델과 합쳐서 모바일용 GGUF 파일로 내보내기
# 'q4_k_m' 양자화 방식 지정
model.save_pretrained_gguf("ticket_model_gguf", tokenizer, quantization_method = "q4_k_m")

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:00<00:00, 13957.75it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:36<00:00, 48.41s/it]


Unsloth: Merge process complete. Saved to `/content/ticket_model_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['ticket_model_gguf_gguf/llama-3.2-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['ticket_model_gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for 

{'save_directory': 'ticket_model_gguf',
 'gguf_directory': 'ticket_model_gguf_gguf',
 'gguf_files': ['ticket_model_gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf'],
 'modelfile_location': 'ticket_model_gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}